Experiment 7: Transfer Learning for Image Classification with Pre-trained
Models
In this experiment, students will apply transfer learning techniques using pretrained
models like VGG16 or ResNet for a new image classification task.

In [ ]:
# ==============================
# 1. Import Libraries
# ==============================
import tensorflow as tf
from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D, Input
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# ==============================
# 2. Load Dataset
# ==============================
(X_train, y_train), (X_test, y_test) = cifar10.load_data()

# Convert labels
y_train = to_categorical(y_train, 10)
y_test = to_categorical(y_test, 10)

# ==============================
# 3. Data Augmentation + Preprocessing
# ==============================
train_gen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True
)

test_gen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

train_data = train_gen.flow(X_train, y_train, batch_size=32)
test_data = test_gen.flow(X_test, y_test, batch_size=32)

# ==============================
# 4. Load Pretrained Model
# ==============================
base_model = VGG16(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

# Freeze ALL layers initially
for layer in base_model.layers:
    layer.trainable = False

# ==============================
# 5. Custom Classifier Head
# ==============================
inputs = Input(shape=(32, 32, 3))

x = tf.keras.layers.Resizing(224, 224)(inputs)
x = base_model(x, training=False)
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.5)(x)
outputs = Dense(10, activation='softmax')(x)

model = Model(inputs, outputs)

# ==============================
# 6. Compile (Phase 1)
# ==============================
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# ==============================
# 7. Train (Feature Extraction)
# ==============================
history = model.fit(
    train_data,
    epochs=10,
    validation_data=test_data
)

# ==============================
# 8. Fine-Tuning (Phase 2)
# ==============================
# Unfreeze last few layers
for layer in base_model.layers[-6:]:
    layer.trainable = True

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history_fine = model.fit(
    train_data,
    epochs=5,
    validation_data=test_data
)

# ==============================
# 9. Evaluate
# ==============================
loss, accuracy = model.evaluate(test_data)
print("Final Test Accuracy:", accuracy)

In [ ]:
Epoch 1/10
1563/1563 [==============================] - 214s 136ms/step - loss: 1.5390 - accuracy: 0.5118 - val_loss: 0.7347 - val_accuracy: 0.7518
Epoch 2/10
1563/1563 [==============================] - 212s 136ms/step - loss: 0.9269 - accuracy: 0.6800 - val_loss: 0.6173 - val_accuracy: 0.7887
Epoch 3/10
1563/1563 [==============================] - 212s 135ms/step - loss: 0.8073 - accuracy: 0.7204 - val_loss: 0.5639 - val_accuracy: 0.8086
Epoch 4/10
1563/1563 [==============================] - 212s 135ms/step - loss: 0.7394 - accuracy: 0.7431 - val_loss: 0.5245 - val_accuracy: 0.8211
Epoch 5/10
1563/1563 [==============================] - 212s 136ms/step - loss: 0.7005 - accuracy: 0.7591 - val_loss: 0.5150 - val_accuracy: 0.8250
Epoch 6/10
1563/1563 [==============================] - 211s 135ms/step - loss: 0.6628 - accuracy: 0.7714 - val_loss: 0.4977 - val_accuracy: 0.8322
Epoch 7/10
1563/1563 [==============================] - 211s 135ms/step - loss: 0.6379 - accuracy: 0.7797 - val_loss: 0.4794 - val_accuracy: 0.8386
Epoch 8/10
1563/1563 [==============================] - 211s 135ms/step - loss: 0.6156 - accuracy: 0.7869 - val_loss: 0.4727 - val_accuracy: 0.8391
Epoch 9/10
1563/1563 [==============================] - 212s 135ms/step - loss: 0.5986 - accuracy: 0.7918 - val_loss: 0.4722 - val_accuracy: 0.8378
Epoch 10/10
1563/1563 [==============================] - 211s 135ms/step - loss: 0.5883 - accuracy: 0.7974 - val_loss: 0.4629 - val_accuracy: 0.8413
Epoch 1/5
1563/1563 [==============================] - 274s 175ms/step - loss: 0.4928 - accuracy: 0.8332 - val_loss: 0.3097 - val_accuracy: 0.8961
Epoch 2/5
1563/1563 [==============================] - 273s 175ms/step - loss: 0.3703 - accuracy: 0.8741 - val_loss: 0.2894 - val_accuracy: 0.9081
Epoch 3/5
1563/1563 [==============================] - 273s 175ms/step - loss: 0.3182 - accuracy: 0.8909 - val_loss: 0.2556 - val_accuracy: 0.9184
Epoch 4/5
1563/1563 [==============================] - 271s 173ms/step - loss: 0.2694 - accuracy: 0.9077 - val_loss: 0.2379 - val_accuracy: 0.9251
Epoch 5/5
1563/1563 [==============================] - 273s 175ms/step - loss: 0.2391 - accuracy: 0.9166 - val_loss: 0.2441 - val_accuracy: 0.9219
313/313 [==============================] - 36s 114ms/step - loss: 0.2441 - accuracy: 0.9219
Final Test Accuracy: 0.9218999743461609